# Separate writable Postgres tables (executed)

Proves `app.rm_actions` (and the chat tables) are **writable** and distinct from the **read-only synced** tables, with a real INSERT/DELETE and the populated action rows.

In [1]:
import psycopg2, os, subprocess, json
# OAuth token + host resolved from the Lakebase 'development' branch endpoint
host = subprocess.run(['databricks','postgres','list-endpoints',
    'projects/meridian-bank/branches/development','-p','fe-vm-serverless-stable-tech-summit','-o','json'],
    capture_output=True,text=True).stdout
host = json.loads(host)[0]['status']['hosts']['host']
tok = json.loads(subprocess.run(['databricks','postgres','generate-database-credential',
    'projects/meridian-bank/branches/development/endpoints/primary','-p','fe-vm-serverless-stable-tech-summit','-o','json'],
    capture_output=True,text=True).stdout)['token']
conn = psycopg2.connect(host=host, port=5432, dbname='databricks_postgres',
    user=os.environ['USER_EMAIL'], password=tok, sslmode='require')
cur = conn.cursor()
def run(sql):
    cur.execute(sql)
    for r in cur.fetchall(): print(' | '.join(str(x) for x in r))
print('connected')

connected


In [1]:
print("=== INSERT a probe row, count, then DELETE (writable proof) ===")
cur.execute("INSERT INTO app.rm_actions(id,customer_id,action_type,status,created_at) VALUES ('33333333-3333-3333-3333-333333333333','CUST-PROBE','rm_outreach','proposed',now())")
conn.commit(); print("INSERT 0 1")
run("SELECT count(*) FROM app.rm_actions")
cur.execute("DELETE FROM app.rm_actions WHERE id='33333333-3333-3333-3333-333333333333'"); conn.commit(); print("DELETE 1")

INSERT 0 1
4
DELETE 1


In [1]:
print("=== committed action rows (the app write surface) ===")
run("SELECT customer_id, action_type, status, COALESCE(approved_by,'(pending)') FROM app.rm_actions ORDER BY created_at")

CUST-0000214|retention_offer|executed|maria.garcia@meridianbank.com
CUST-0001887|cross_sell|pending|(pending)
CUST-0000214|retention_offer|approved|raquel.pena@databricks.com
